# KAC-Net v2: Knowledge-Enriched Attentive Contrastive Network

This notebook runs the **KAC-Net v2** model on the 10x Human Lymph Node CITE-seq dataset. KAC-Net v2 integrates spatial multi-omics by combining:
1. **Module 1 & 2**: Knowledge-enriched encoding of RNA (scGPT/PCA fallback) and Centered Log Ratio (CLR) normalization of ADT.
2. **Module 3 & 4**: Multi-graph construction (spatial & feature) with local GATv2 encoders.
3. **Module 5**: Cross-modal contrastive alignment (InfoNCE loss).
4. **Module 6**: Adaptive dual-attention fusion (within and between modality attention layers).
5. **Module 7**: Reconstruction decoders and spatial Laplacian regularization.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path
import torch

from KAC_Net_v2 import (
    Train_KACNet,
    search_res,
    clustering,
    compute_ari,
    plot_spatial_domains,
    plot_umap,
    plot_modality_weights,
    plot_loss_curve
)

plt.style.use('default')
random_seed = 42
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

## Load 10x Human Lymph Node Data

In [ ]:
data_path = Path('./data/10x_human_lymph_node_A1')

adata_rna = sc.read_h5ad(data_path / 'adata_RNA.h5ad')
adata_adt = sc.read_h5ad(data_path / 'adata_ADT.h5ad')
annot = pd.read_csv(data_path / 'annotation.csv', index_col=0)

print(f"Dataset loaded from: {data_path}")
print(f"RNA shape: {adata_rna.shape}")
print(f"ADT shape: {adata_adt.shape}")

df_labels = list(annot['manual-anno'])
label_type = sorted(list(set(df_labels)))
print(f"Cell types: {label_type}")
print(f"Number of cells: {adata_rna.n_obs}")

## Generate Paired Multi-Omics Data with Simulation Noise

In [ ]:
# Prepare Omics-1 (RNA)
adata1 = adata_rna.copy()
adata1.obs['CellType'] = df_labels

# Prepare Omics-2 (ADT)
adata2 = adata_adt.copy()
adata2.obs['CellType'] = df_labels

# Add simulation noise to create heterogeneity (matching the COSMOS experiment)
if len(label_type) >= 2:
    index_int1 = np.where(np.array(df_labels) == label_type[0])[0]
    index_int2 = np.where(np.array(df_labels) == label_type[1])[0]
else:
    index_int1 = np.arange(0, adata1.n_obs // 2)
    index_int2 = np.arange(adata1.n_obs // 2, adata1.n_obs)

print(f"Index set 1 size: {len(index_int1)}")
print(f"Index set 2 size: {len(index_int2)}")

# Shuffle cells to create heterogeneity
np.random.seed(random_seed)
perm_idx1 = np.random.permutation(index_int1)
adata1.X[index_int1, :] = adata1.X[perm_idx1, :]

np.random.seed(random_seed + 1)
perm_idx2 = np.random.permutation(index_int2)
adata2.X[index_int2, :] = adata2.X[perm_idx2, :]

print(f"\n✓ Created paired heterogeneous omics data for KAC-Net")

## Run KAC-Net v2 Pipeline

In [ ]:
print("Initializing KAC-Net v2...")
# Note: set scgpt_model_dir to a valid path if you want to use the pre-trained scGPT model,
# otherwise the model falls back to a PCA mock embedding automatically.
trainer = Train_KACNet(
    adata_rna=adata1,
    adata_adt=adata2,
    scgpt_model_dir=None, 
    latent_dim=128,
    proj_dim=64,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=300,
    patience=20,
    weights=[1.0, 5.0, 1.0, 1.0],  # [w_recon_rna, w_recon_adt, w_align, w_laplacian]
    temperature=0.07,
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    random_seed=random_seed
)

results = trainer.train()

## Calculate ARI Scores

In [ ]:
embedding = results['h_fused']

print("Screening Leiden clustering resolutions to find optimal ARI...")
best_res, best_ari, best_clusters = search_res(
    embedding, 
    ground_truth_labels=df_labels,
    start=0.1, 
    end=1.0, 
    step=0.05,
    random_seed=random_seed
)

print(f"\n✓ Optimal KAC-Net Clustering Result:")
print(f"  ARI Score: {best_ari:.4f}")
print(f"  Resolution: {best_res:.2f}")
print(f"  Clusters: {len(set(best_clusters))}")

## Visualizations

In [ ]:
# 1. Plot Loss Convergence Curve
plot_loss_curve(trainer.loss_history)

# 2. Plot Modality Gate Weights distribution
plot_modality_weights(results['gate_weights'])

# 3. Plot Spatial Domains mapping back to slide coordinates
adata_viz = adata1.copy()
adata_viz.obs['KACNet_Clusters'] = best_clusters.astype(str)
plot_spatial_domains(adata_viz, 'KACNet_Clusters')

# 4. Plot UMAP representations
plot_umap(embedding, df_labels, title="KAC-Net Latent Space (Manual Annotations)")
plot_umap(embedding, best_clusters, title="KAC-Net Latent Space (Leiden Clusters)")